# Step 2 · Build the labour market page

**What this does:** reads the downloaded series (`store/statcan.csv`), works out everything the labour page shows,
and writes it to one file, `../site/data/labour.json`. The page only draws what is in that file.
It also writes `../site/tile.json`, the few numbers the labour tile on the Monitor front page shows.

**What goes into the file:**
1. the four numbers at the top of the page (*where things stand*),
2. a few sentences on what the latest release changed,
3. seven charts in four sections, each with a question as its title and a reading written from the numbers:

| Section | Charts |
|---|---|
| Where is unemployment? | unemployment rate, with what moved it underneath (month or 12 months) · provinces |
| How are jobs changing? | change by industry over the last month, the year to date and 12 months |
| How hard is it to find work? | jobseekers per vacancy · length of unemployment |
| Who is unemployed? | each age group against its own history, by education · the same over time |

All the text is written by rules in this notebook ("the highest since…", "up 0.2 points…"), not by hand and not by AI,
so it is always consistent with the data.

**Run after:** `01_fetch_statcan.ipynb`.

In [ ]:
import json                     # to write the output file
from datetime import datetime   # for the time the file was built
import numpy as np              # logarithms, for the unemployment breakdown
import pandas as pd             # tables

STATCAN_TABLE = "https://www150.statcan.gc.ca/t1/tbl1/en/tv.action?pid="   # + table number without dashes

## 1. Load the data

`store/statcan.csv` is long: one row per series per period. We turn it into a wide table, `w`, with one column per
series and one row per month, so that `w["unemp_rate"]` is the unemployment rate over time. Quarterly series
(productivity) sit on the first month of each quarter.

In [ ]:
data = pd.read_csv("store/statcan.csv", parse_dates=["date"])
series = pd.read_csv("../series.csv")

w = data.pivot(index="date", columns="id", values="value")      # wide: periods down, series across

# when each series was last published by StatCan (the release time of its latest point)
release_time = data.sort_values("date").groupby("id")["release_time"].last()

# the latest month of the Labour Force Survey: the month this page is "about"
lfs_month = w["unemp_rate"].dropna().index[-1]
print("Latest LFS month:", lfs_month.strftime("%B %Y"), "| released", release_time["unemp_rate"])

## 2. Small helpers for writing text

Short functions, because the same wording is needed in many places:
- `month_name` turns a date into "September 2026" (or "Sep 2026" when short); `quarter_name` gives "Q2 2026",
- `signed` writes a change with its sign: +0.2 or −0.2 (a real minus sign),
- `since` finds how unusual the latest value is: "the highest since May 2022",
- `percentile` says where a value sits in a series' own history, and `ordinal` writes it as "29th",
- `points` turns a series into the list of [period, value] pairs the page draws (its whole history unless a start is given).

In [ ]:
def month_name(date, short=False):
    return date.strftime("%b %Y" if short else "%B %Y")


def quarter_name(date):
    return f"Q{(date.month - 1) // 3 + 1} {date.year}"


def signed(x, decimals=1):
    text = f"{abs(x):,.{decimals}f}"
    if round(x, decimals) == 0:
        return text                       # no sign on zero
    return ("+" if x > 0 else "\u2212") + text


def since(s):
    # "the highest since ..." / "the lowest since ..." for the latest value of series s.
    # Returns "" when the latest value is not unusual (the same level was seen within the last 12 months).
    s = s.dropna()
    last, before = s.iloc[-1], s.iloc[:-1]
    if last > before.iloc[-1]:
        word, earlier = "highest", before[before >= last]
    elif last < before.iloc[-1]:
        word, earlier = "lowest", before[before <= last]
    else:
        return ""
    if earlier.empty:
        return f"the {word} since the data begin in {s.index[0].year}"
    when = earlier.index[-1]
    if (s.index[-1] - when).days < 365:
        return ""
    text = f"the {word} since {month_name(when, short=True)}"
    # if that date falls in the pandemic years, also say what it is outside them
    if pd.Timestamp("2020-03-01") <= when <= pd.Timestamp("2021-12-01"):
        outside = earlier[(earlier.index < "2020-03-01") | (earlier.index > "2021-12-01")]
        if outside.empty:
            text += ", and outside the pandemic since the data begin"
        else:
            text += f", or {month_name(outside.index[-1], short=True)} outside the pandemic"
    return text


def percentile(s, value):
    # share of all months in the series' history with a lower value, 0 to 100
    s = s.dropna()
    return round((s < value).mean() * 100)


def ordinal(n):
    suffix = "th" if 10 <= n % 100 <= 20 else {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")
    return f"{n}{suffix}"


def points(s, start=None, decimals=1, gaps=False):
    # [["2000-01", 6.8], ...]. With gaps=True, months missing inside the series come out as None,
    # so the page breaks the line there instead of drawing straight across.
    if start:
        s = s[s.index >= start]
    if gaps:
        s = s.loc[s.first_valid_index():s.last_valid_index()]
        s = s.reindex(pd.date_range(s.index[0], s.index[-1], freq="MS"))
    else:
        s = s.dropna()
    return [[d.strftime("%Y-%m"), None if pd.isna(v) else round(float(v), decimals)] for d, v in s.items()]


print(month_name(lfs_month), "|", signed(-0.24), "|", ordinal(22), ordinal(13), "|", since(w["unemp_rate"]) or "(not unusual)")

## 3. Series we calculate

Some charts need series that StatCan does not publish directly:

- **12-month growth** of wages and prices (per cent change from the same month a year earlier).
- **Unemployed people per job vacancy**: LFS unemployed (thousands, so ×1,000) divided by job vacancies.
- **Long unemployment**: the share of unemployed people looking for 52 weeks or more (StatCan publishes the 27-weeks
  share directly).
- **What moved the unemployment rate: jobs, population and participation.** The employed are a share of the labour
  force, and the labour force is the population times the participation rate:

  `1 − unemployment rate = employment ÷ (population × participation rate)`

  So when the unemployment rate changes, it is because of one or more of three things:
  - **jobs**: more people employed pushes unemployment down,
  - **population**: more people of working age, with the same number of jobs, pushes it up,
  - **participation**: a larger share of people working or looking pushes it up; people giving up on looking push it down.

  In logarithms the identity becomes a sum, so each part is its own log change, multiplied by `1 − unemployment rate`
  (averaged over the two months) to turn it into percentage points. The three parts add up to the actual change to
  within a few hundredths of a point (checked below).

  Everything here uses the counts (thousands of people), not the published rates, because those are rounded to one
  decimal: a monthly change of 0.1 points would otherwise be mostly rounding. It is done twice: over one month (what
  moved the latest reading) and over 12 months.

In [ ]:
growth = lambda s, n=12: (s / s.shift(n) - 1) * 100     # per cent change over n periods

w["wage_growth"] = growth(w["wage_hourly"])
w["inflation"] = growth(w["cpi"])
w["unemp_per_vacancy"] = w["unemployment"] * 1000 / w["vacancies"]
w["ltu52_share"] = w["unemployed_52w"] / w["unemployed_all_dur"] * 100

# --- what moved the unemployment rate ---
E, U, P = w["employment"], w["unemployment"], w["population"]    # thousands of people, seasonally adjusted
LF = E + U                                                       # labour force
w["ur_exact"] = U / LF * 100                                     # the unemployment rate from the counts
w["pr_exact"] = LF / P * 100                                     # the participation rate from the counts

breakdown = {}
for lag, name in [(1, "month"), (12, "year")]:
    keep = 1 - (w["ur_exact"] + w["ur_exact"].shift(lag)) / 200          # 1 - unemployment rate, averaged
    breakdown[name] = pd.DataFrame({
        "jobs": -keep * 100 * np.log(E / E.shift(lag)),
        "population": keep * 100 * np.log(P / P.shift(lag)),
        "participation": keep * 100 * np.log(w["pr_exact"] / w["pr_exact"].shift(lag)),
        "total": w["ur_exact"] - w["ur_exact"].shift(lag),
    })
    parts = breakdown[name][["jobs", "population", "participation"]].sum(axis=1)
    gap = (parts - breakdown[name]["total"]).abs()
    print(f"{name}: the parts miss the actual change by at most {gap.loc['1977':'2019'].max():.3f} points before 2020, "
          f"{gap.max():.3f} in the pandemic")

breakdown["month"].tail(3).round(2)

## 4. Where things stand: the four numbers at the top

1. the unemployment rate and its change from last month,
2. the monthly change in jobs, and the change over the past 12 months,
3. wage growth against inflation,
4. how many provinces saw unemployment rise from last month, with the highest and lowest rate.

In [ ]:
lfs = w.loc[:lfs_month]
now, prev, year_ago = lfs.iloc[-1], lfs.iloc[-2], lfs.iloc[-13]
Z = 1.645                                                # 90% confidence: a change beyond Z standard errors counts as real

# 1. unemployment rate
ur_change = now["unemp_rate"] - prev["unemp_rate"]

# 2. jobs: the monthly change, and the change over the past 12 months
jobs_change = (now["employment"] - prev["employment"]) * 1000
jobs_12 = (now["employment"] - year_ago["employment"]) * 1000

# 3. wages against inflation, in the latest month both exist
both = w[["wage_growth", "inflation"]].dropna()
wi_month = both.index[-1]
wage_now, infl_now = both.iloc[-1]
real = wage_now - infl_now

# 4. provinces where the unemployment rate rose from last month (the published rates, to one decimal)
PROVINCES = [("ur_nl", "N.L."), ("ur_pe", "P.E.I."), ("ur_ns", "N.S."), ("ur_nb", "N.B."), ("ur_qc", "Que."),
             ("ur_on", "Ont."), ("ur_mb", "Man."), ("ur_sk", "Sask."), ("ur_ab", "Alta."), ("ur_bc", "B.C.")]
moves = lfs[[col for col, _ in PROVINCES]].round(1).diff().round(1)
rose = (moves > 0).sum(axis=1)                                     # each month: how many provinces' rates rose
rates_now = {name: now[col] for col, name in PROVINCES}
highest, lowest = max(rates_now, key=rates_now.get), min(rates_now, key=rates_now.get)

spark = lambda s: [round(float(v), 2) for v in s.dropna().iloc[-36:]]    # last 3 years, for the small trend lines

tiles = [
    {"label": "Unemployment rate", "value": f"{now['unemp_rate']:.1f}%",
     "change": f"{signed(ur_change)} pts from {lfs.index[-2].strftime('%B')}",
     "spark": spark(w["unemp_rate"])},
    {"label": f"Jobs in {lfs_month.strftime('%B')}", "value": signed(jobs_change / 1000, 0) + "k",
     "change": f"{signed(jobs_12 / 1000, 0)}k over the past 12 months",
     "spark": spark(w["employment"])},
    {"label": "Wages vs a year ago", "value": f"{signed(wage_now)}%",
     "change": f"inflation {infl_now:.1f}% ({month_name(wi_month, short=True)})",
     "spark": spark(w["wage_growth"])},
    # ("Unemployment rose in 5 provinces": the number large, the word after it small)
    {"label": "Unemployment rose in", "value": str(rose.iloc[-1]), "unit": "province" if rose.iloc[-1] == 1 else "provinces",
     # (short enough for one line in a desktop tile, even for "high P.E.I. 10.5% · low Sask. 10.5%")
     "change": f"high {highest} {rates_now[highest]:.1f}% \u00b7 low {lowest} {rates_now[lowest]:.1f}%",
     "spark": spark(rose.iloc[1:])},
]
for t in tiles:
    print(f"{t['label']:28} {t['value']:>8}   {t['change']}")

## 5. Who is unemployed: each age group against its own history, by education

This follows Gad Levanon (*Labor Matters*): rather than comparing unemployment rates across groups (young people are
always higher), each group is compared with **its own history**, as a percentile. *The 50th percentile means normal;
higher means worse.*

How, step by step:
1. **Groups.** StatCan's LFS table by educational attainment (14-10-0019) has broad age bands only. We use 15 to 24,
   25 to 44, 45 to 54 (not published: the 25 to 54 counts minus the 25 to 44 counts) and 55 to 64, plus all of them
   together (15 to 64: 15 to 24 + 25 to 54 + 55 to 64, a suppressed part counting as zero), and five education
   groups. A group's counts are the sum of StatCan's levels in it (e.g. "High school or less" = 0 to 8 years + some
   high school + high school graduate). A month or two that StatCan suppressed in one of those levels (it does this
   when a number is very small) is filled from the months on either side.
2. **Rate, then average.** Each month's unemployment rate is the group's unemployed divided by its labour force. The
   table is not seasonally adjusted, so we take the **12-month average of that monthly rate**.
3. **Rank.** The latest 12-month average is ranked against every month's 12-month average **since January 2000**:
   the percentile is the share of those months with a lower rate. The same ranking, month by month, gives the heat
   map over time.
4. **Too small to measure.** A group averaging fewer than 10,000 unemployed people over the last year (15 to 24 year
   olds with a degree above a bachelor's) is left out: its rate moves by whole points from survey noise alone.

In [ ]:
WINDOW = "2000-01-01"
edu_groups = {"all": ["all"], "hs": ["0_8", "somehs", "hsgrad"], "college": ["someps", "pscert"],
              "bach": ["bach"], "abovebach": ["abovebach"]}
edu_labels = {"all": "All education levels", "hs": "High school or less", "college": "College, trades or some postsecondary",
              "bach": "Bachelor's degree", "abovebach": "Above bachelor's degree"}
edu_short = {"all": "all education levels", "hs": "high school or less", "college": "college, trades or some postsecondary",
             "bach": "a bachelor's degree", "abovebach": "a degree above a bachelor's"}
age_labels = {"15_24": "15 to 24", "25_44": "25 to 44", "45_54": "45 to 54", "55_64": "55 to 64", "15_64": "15 to 64"}
edu_tab = {"all": "All", "hs": "High school or less", "college": "College or trades", "bach": "Bachelor's", "abovebach": "Above bachelor's"}   # short, for tabs and legends


def counts(kind, group, age):
    # thousands of people (kind "u" = unemployed, "lf" = labour force) in one education group and age band
    if age == "45_54":                       # not published: 25 to 54 minus 25 to 44
        return counts(kind, group, "25_54") - counts(kind, group, "25_44")
    if age == "15_64":                       # all ages shown: 15 to 24 + 25 to 54 + 55 to 64; a part StatCan suppressed
        # (fewer than 1,500 people, e.g. unemployed 15 to 24 year olds above a bachelor's) counts as zero
        return sum(counts(kind, group, a).fillna(0) for a in ["15_24", "25_54", "55_64"])
    total = 0
    for e in edu_groups[group]:
        # a month or two that StatCan suppressed (too small) inside a series is filled from its neighbours
        total = total + w[f"edu_{kind}_{e}_{age}"].interpolate(limit=2, limit_area="inside")
    return total


history = {}    # (group, age) -> 12-month average rate since 2000, or None when the group is too small
for g in edu_groups:
    for a in age_labels:
        u, lf = counts("u", g, a), counts("lf", g, a)
        rate = u / lf * 100                              # each month's rate first ...
        avg12 = rate.rolling(12).mean()                  # ... then its 12-month average
        hist = avg12.loc[WINDOW:lfs_month]
        too_small = u.loc[:lfs_month].iloc[-12:].mean() < 10 or hist.isna().any()
        history[(g, a)] = None if too_small else hist

# percentile of every month's value within the group's own history since 2000 (share of months lower)
pctile = {k: None if h is None else ((h.rank(method="min") - 1) / len(h) * 100) for k, h in history.items()}

table = pd.DataFrame({age_labels[a]: [None if pctile[(g, a)] is None else round(pctile[(g, a)].iloc[-1]) for g in edu_groups]
                      for a in age_labels}, index=[edu_labels[g] for g in edu_groups])
print("Percentile now (12-month average, since 2000):")
table

## 6. What the latest release changed

Five sentences on the latest Labour Force Survey, written from the numbers, one for each part of the page: the
unemployment rate and what moved it, jobs, how hard it is to find work, who is worst off, and wages.

In [ ]:
changes = []
m = breakdown["month"].loc[lfs_month]

def drivers(b):
    # the three parts in words, one line for what pushed the unemployment rate up and one for what pulled it down.
    # Each line says its direction in words (and the page colours it), so "+0.30" next to "job losses" can't be
    # mistaken for good news. ("participation in job market": checked to keep the reading to three lines in every
    # combination of directions, at the narrowest width where two cards sit side by side.)
    up_words = {"jobs": "job losses", "population": "population growth", "participation": "participation in job market"}
    down_words = {"jobs": "job gains", "population": "population decline", "participation": "participation in job market"}
    keys = ["jobs", "population", "participation"]
    ups = [f"{up_words[k]} {signed(b[k], 2)}" for k in keys if round(b[k], 2) > 0]
    downs = [f"{down_words[k]} {signed(b[k], 2)}" for k in keys if round(b[k], 2) < 0]
    lines = []
    if ups:
        lines.append({"label": "Pushed it up:", "text": ", ".join(ups), "tone": "up"})
    if downs:
        lines.append({"label": "Pulled it down:", "text": ", ".join(downs), "tone": "down"})
    return lines

# 1. the unemployment rate, and what moved it (in words, as on the chart)
moved = "rose" if ur_change > 0 else "fell" if ur_change < 0 else "held"
text = (f"The unemployment rate {moved} {abs(ur_change):.1f} points to {now['unemp_rate']:.1f}% in {month_name(lfs_month)}."
        if moved != "held" else f"The unemployment rate held at {now['unemp_rate']:.1f}% in {month_name(lfs_month)}.")
changes.append(text + "".join(f" {line['label']} {line['text']}." for line in drivers(m)))

# 2. jobs: the month (full- and part-time) and the past 12 months
ft = (now["fulltime"] - prev["fulltime"]) * 1000
pt = (now["parttime"] - prev["parttime"]) * 1000
changes.append(
    f"Employment {'rose' if jobs_change > 0 else 'fell'} by {abs(jobs_change) / 1000:,.0f}k in {lfs_month.strftime('%B')} "
    f"(full-time {signed(ft / 1000, 0)}k, part-time {signed(pt / 1000, 0)}k); over the past 12 months it "
    f"{'rose' if jobs_12 > 0 else 'fell'} by {abs(jobs_12) / 1000:,.0f}k."
)

# 3. finding work: jobseekers per vacancy (the vacancy survey runs about two months behind) and long searches
v = w["unemp_per_vacancy"].dropna()
long_search = lfs["ltu_share"].rolling(3).mean()           # looking for 27 weeks or more, 3-month average (as on the chart)
changes.append(
    f"There were {v.iloc[-1]:.1f} unemployed people for every job vacancy in {month_name(v.index[-1])} ({v.iloc[-13]:.1f} a year "
    f"earlier), and {long_search.iloc[-1]:.1f}% of unemployed people had been looking for over 6 months "
    f"({long_search.iloc[-13]:.1f}% a year earlier)."
)

# 4. the age and education group worst off against its own history
worst = max(((g, a) for (g, a), p in pctile.items() if p is not None and g != "all" and a != "15_64"), key=lambda k: pctile[k].iloc[-1])
g, a = worst
changes.append(
    f"Against their own history since 2000, {age_labels[a]} year olds with {edu_short[g]} are the worst off: "
    f"{history[worst].iloc[-1]:.1f}% unemployed (12-month average), the {ordinal(round(pctile[worst].iloc[-1]))} percentile."
)

# 5. real wages
changes.append(
    f"Average hourly wages were up {wage_now:.1f}% on a year earlier and consumer prices {infl_now:.1f}% ({month_name(wi_month)}): "
    + (f"real wages up {real:.1f}%." if real > 0.05 else f"real wages down {abs(real):.1f}%." if real < -0.05 else "real wages flat.")
)

for c in changes:
    print("-", c)

## 7. The charts

Seven charts in four sections. Each chart is a small dictionary: `title` (the question it answers), `reading` (one
sentence on what it shows now), `kind` (how the page draws it), the data, a `note` saying what the chart is, and
`source` / `source_url`.

Kinds used here: `line` (time series; optional `lower` panel underneath, sharing the time axis, with `modes` the
reader switches between), `compare` (each row now, the previous month and a year ago), `bars_multi` (horizontal
bars, one column per period) and `cohort_range` / `cohort_history` (two linked charts: clicking a row or an education
tab on the first changes the second).

Colours are named by role, and the page picks the actual colours: `main` (the portfolio blue), `second` (orange),
`third` (teal), `fourth` (purple) and `context` (grey).

In [ ]:
def source(*tables):
    # "Statistics Canada, table 14-10-0287-01" plus the link to the table; with several tables
    # ("tables 14-10-0287-01 & 14-10-0432-01") each table number links to its own table, through source_links
    url = lambda t: STATCAN_TABLE + t.replace("-", "")[:8] + "01"
    names = ", ".join(tables[:-1]) + " & " + tables[-1] if len(tables) > 1 else tables[0]
    out = {"source": "Statistics Canada, table" + ("s " if len(tables) > 1 else " ") + names, "source_url": url(tables[0])}
    if len(tables) > 1:
        out["source_links"] = {"text": "Statistics Canada, tables ", "links": [[t, url(t)] for t in tables]}
    return out

LFS = source("14-10-0287-01")
sections = []

### 7.1 Where is unemployment?

The unemployment rate since 1976, with a panel underneath splitting each month's change into jobs, population and
participation (section 3). The panel has a fixed axis of ±1 point, because the pandemic months (up to 5 points in a
month) would flatten everything else.

The chart's title says where the rate went ("Unemployment rose to 6.5% in September 2026"); the reading gives its
percentile since 2000 and the three parts, grouped by what they did ("Pushed it up: job losses 0.30, …").

Next to it, each province now and a year ago, with its change from last month in a chip.

In [ ]:
charts = []

# --- the unemployment rate, and what moved it this month ---
pct = percentile(w["unemp_rate"].loc[WINDOW:], now["unemp_rate"])
direction = "rose" if ur_change > 0 else "declined" if ur_change < 0 else "held"
title = f"Unemployment {direction} {'at' if direction == 'held' else 'to'} {now['unemp_rate']:.1f}% in {month_name(lfs_month)}"
# the percentile first, then a line for what pushed the rate up and one for what pulled it down
reading_lines = [{"text": f"That puts it at the {ordinal(pct)} percentile of its history since 2000."}] + drivers(m)
reading = "\n".join((line.get("label", "") + " " + line["text"]).strip() for line in reading_lines)

b = breakdown["month"]
charts.append({
    "id": "unemployment", "title": title,
    "reading": reading, "reading_lines": reading_lines,
    "kind": "line", "unit": "%", "decimals": 1,
    "series": [{"name": "Unemployment rate", "color": "main", "mark": "line", "data": points(w["unemp_rate"])}],
    "lower": {"unit": "pts", "decimals": 2, "clip": [-1, 1], "ticks": [-1, 0, 1],
              "label": "What moved it: percentage points change from the previous month",
              # (colours checked for colour-blind readers, against each other and against the blue line above)
              "series": [{"name": "Jobs", "color": "teal", "data": points(b["jobs"], decimals=2)},
                         {"name": "Population", "color": "gold", "data": points(b["population"], decimals=2)},
                         {"name": "Participation", "color": "lavender", "data": points(b["participation"], decimals=2)}],
              # the total change is listed in the tooltip but not drawn: the title already says which way the rate moved
              "total": {"name": "Total change", "line": False, "data": points(b["total"], decimals=2)}},
    "note": "Unemployment rate, ages 15 and over, seasonally adjusted.",
    **LFS,
})

# --- provinces: now, the previous month and a year ago (and Canada, in its place by rate, in bold) ---
provinces = PROVINCES                                     # (defined with the tiles, section 4)
rows = []
for col, name in provinces:
    s = w[col].loc[:lfs_month]
    # 3-month trend: the average of the last three months against the average of the three before (e.g. Jul-Sep vs
    # Apr-Jun). Steadier than one month: provincial samples are small, so single months are noisy.
    trend3 = s.iloc[-3:].mean() - s.iloc[-6:-3].mean()
    rows.append({"label": name, "now": round(float(now[col]), 1), "prev": round(float(prev[col]), 1), "year": round(float(year_ago[col]), 1),
                 "m1": round(float(now[col] - prev[col]), 1), "m3": round(float(trend3), 1)})
up_year = sum(d["now"] > d["year"] for d in rows)
up_month = sum(round(d["now"] - d["prev"], 1) > 0 for d in rows)       # rose from last month
down_month = sum(round(d["now"] - d["prev"], 1) < 0 for d in rows)     # declined from last month
# say whichever way most provinces moved ("declined in 6 provinces"); on a tie, both
if up_month > down_month:
    province_sentence = f"Unemployment rose in {up_month} provinces in {month_name(lfs_month)}."
elif down_month > up_month:
    province_sentence = f"Unemployment declined in {down_month} provinces in {month_name(lfs_month)}."
else:
    province_sentence = f"Unemployment rose in {up_month} provinces and declined in {down_month} in {month_name(lfs_month)}."
s = w["unemp_rate"].loc[:lfs_month]
rows.append({"label": "Canada", "now": round(float(now["unemp_rate"]), 1), "prev": round(float(prev["unemp_rate"]), 1),
             "year": round(float(year_ago["unemp_rate"]), 1), "m1": round(float(now["unemp_rate"] - prev["unemp_rate"]), 1),
             "m3": round(float(s.iloc[-3:].mean() - s.iloc[-6:-3].mean()), 1)})
rows.sort(key=lambda d: d["now"], reverse=True)
charts.append({
    "id": "provinces", "title": "Where is unemployment highest?",
    "reading": province_sentence,
    "kind": "compare", "unit": "%", "rows": rows, "highlight": "Canada",
    "labels": {"now": month_name(lfs_month, short=True), "year": "A year ago", "prev": "vs " + prev.name.strftime("%b")},
    # the chip columns, right of the chart: change from last month, and the 3-month trend
    "chips": [{"key": "m1", "label": "vs " + prev.name.strftime("%b")}, {"key": "m3", "label": "3-mo trend"}],
    "fill": True,                                            # stretch to the height of the card next to it
    "note": "Unemployment rate, Canada and provinces, ages 15 and over, seasonally adjusted.", **LFS,
})

sections.append({"title": "Where is unemployment?", "charts": charts})
print(title); print(reading)

### 7.2 How are jobs changing?

Employment change by industry over three periods side by side: the last month, the year to date (since December) and
the last 12 months.

**Margin of error.** StatCan publishes, for each industry, the standard error of the monthly change and of the 12-month
change. A change larger than 1.645 standard errors is statistically significant at the 90% level; the page outlines
its bar.

For the year to date there is no published standard error, so we estimate one. LFS households stay in the survey for
six months, so two months a few months apart share part of their sample, and the error of the change grows with the
gap until the samples no longer overlap (six months or more apart), after which it is about the same as the 12-month
error. So: one month since December uses the monthly error; six or more use the 12-month error; in between we
interpolate the variance (the squared error) in a straight line, since the shared households lower the variance in
proportion to how many there are. It is an approximation, good enough to tell noise from signal.

In [ ]:
charts = []
k = lfs_month.month                                        # months since December
december = lfs_month - pd.DateOffset(months=k)
a_year_before = lfs_month - pd.DateOffset(months=12)
share = min(k - 1, 5) / 5                                  # 0 for January ... 1 from June on

industries = series[(series["table"] == "14-10-0355-01") & (series["id"].str.startswith("emp_"))
                    & (~series["id"].str.contains("_se_"))]
rows = []
for _, r in industries.iterrows():
    sid = r["id"]
    name = "All industries" if sid == "emp_total" else r["name"]
    level = w[sid]
    se_mm, se_yy = w[sid + "_se_mm"].loc[lfs_month], w[sid + "_se_yy"].loc[lfs_month]
    se_ytd = (se_mm ** 2 + (se_yy ** 2 - se_mm ** 2) * share) ** 0.5     # straight line in the variance, not the error
    changes_k = [level[lfs_month] - level[prev.name], level[lfs_month] - level[december], level[lfs_month] - level[a_year_before]]
    errors = [se_mm, se_ytd, se_yy]
    rows.append({"label": name, "total": sid == "emp_total",
                 "values": [round(c * 1000) for c in changes_k],
                 "significant": [bool(abs(c) > Z * e) for c, e in zip(changes_k, errors)]})
rows.sort(key=lambda r: (not r["total"], -r["values"][2]))        # all industries first, then by the 12-month change
inds = [r for r in rows if not r["total"]]
top = max(inds, key=lambda r: r["values"][2]); bottom = min(inds, key=lambda r: r["values"][2])
total_yy = next(r for r in rows if r["total"])["values"][2]
total_text = (f"grew by {total_yy / 1000:,.0f}k jobs" if total_yy > 0 else f"fell by {-total_yy / 1000:,.0f}k jobs" if total_yy < 0
              else "was unchanged")
charts.append({
    "id": "by_industry", "title": "Which industries are adding and losing jobs?", "wide": True,
    "reading": f"Over the last 12 months, employment {total_text}. {top['label']} added the most jobs ({signed(top['values'][2] / 1000, 0)}k) and "
               f"{bottom['label'].lower()} lost the most ({signed(bottom['values'][2] / 1000, 0)}k).",
    "kind": "bars_multi", "unit": "jobs", "rows": rows, "sort": 2, "level": "90%",     # sort: the column that orders the rows at first (readers can pick another)
    "columns": [{"label": "Last month", "sub": f"{month_name(prev.name, short=True)} to {month_name(lfs_month, short=True)}"},
                {"label": "Year to date", "sub": f"{month_name(december, short=True)} to {month_name(lfs_month, short=True)}"},
                {"label": "Past 12 months", "sub": f"{month_name(a_year_before, short=True)} to {month_name(lfs_month, short=True)}"}],
    "note": "Change in employment by industry, seasonally adjusted. Outlined bars: statistically significant at the 90% level.",
    **source("14-10-0355-01"),
})

sections.append({"title": "How are jobs changing?", "charts": charts})

### 7.3 How hard is it to find work?

Jobseekers per open job, and how long unemployed people have been looking (full history since 1976). The vacancy
survey was suspended from April to September 2020; the page leaves that stretch blank.

In [ ]:
charts = []

# --- unemployed people per vacancy ---
v = w["unemp_per_vacancy"].dropna()
v_low, v_low_date = v.min(), v.idxmin()
charts.append({
    "id": "per_vacancy", "title": "How many jobseekers are there for each open job?",
    "reading": f"{v.iloc[-1]:.1f} unemployed people for every vacancy in {month_name(v.index[-1])}, "
               f"against {v.iloc[-13]:.1f} a year earlier and {v_low:.1f} at the tightest point, {month_name(v_low_date, short=True)}.",
    "kind": "line", "unit": "", "decimals": 2,
    "series": [{"name": "Unemployed people per vacancy", "color": "main", "mark": "line",
                "data": points(w["unemp_per_vacancy"], decimals=2, gaps=True)}],
    "note": "Unemployed people per job vacancy, seasonally adjusted.",
    **source("14-10-0287-01", "14-10-0432-01"),
})

# --- length of unemployment ---
avg3 = lambda s: s.rolling(3).mean()                       # 3-month average: the month and the two before
charts.append({
    "id": "duration", "title": "Are people staying unemployed longer?",
    "reading": f"In {month_name(lfs_month)}, {avg3(lfs['ltu_share']).iloc[-1]:.1f}% of unemployed people had been looking for over 6 months, "
               f"and {avg3(lfs['ltu52_share']).iloc[-1]:.1f}% for over a year.",
    "kind": "line", "unit": "%", "decimals": 1,
    # the 3-month averages are the lines; the monthly values sit faded behind them (context: no end label, end dot,
    # legend entry or tooltip row)
    "series": [{"name": "27 weeks or more, monthly", "color": "main", "mark": "line", "opacity": 0.3, "context": True,
                "data": points(w["ltu_share"])},
               {"name": "52 weeks or more, monthly", "color": "second", "mark": "line", "opacity": 0.3, "context": True,
                "data": points(w["ltu52_share"])},
               {"name": "27 weeks or more", "color": "main", "mark": "line", "data": points(avg3(w["ltu_share"]))},
               {"name": "52 weeks or more", "color": "second", "mark": "line", "data": points(avg3(w["ltu52_share"]))}],
    "note": "Share of unemployed people by time spent looking for work, ages 15 and over, seasonally adjusted, 3-month moving average.",
    **source("14-10-0342-01"),
})

sections.append({"title": "How hard is it to find work?", "charts": charts})

### 7.4 Who is unemployed?

Two charts built from section 5, linked on the page (hovering a group in one lights it up in the other):
- **Now**: two grids, ages down the side and education across, with an "All ages" row and an "All" column. The
  first grid is the unemployment rate (12-month average), the second its percentile against the group's own history.
- **Over time**: a heat map, one row per group and one cell per month since 2000, coloured by that month's percentile
  (in ten steps of ten: blue below 50, orange above). It follows the page's 5-year / 10-year / All buttons.

In [ ]:
charts = []
bands = [a for a in age_labels if a != "15_64"]                    # the four age bands
groups_e = [g for g in edu_groups if g != "all"]                    # the four education groups
back12 = -13                                                        # (position of the same month a year earlier)

# --- now: two grids, the rate and its percentile, for every age band and all ages, every education group and all ---
cells = {}
for g in edu_groups:
    cells[g] = {}
    for a in age_labels:
        h = history[(g, a)]
        cells[g][a] = None if h is None else {
            "rate": round(float(h.iloc[-1]), 1), "pct": int(round(pctile[(g, a)].iloc[-1])),
            "rate_year": round(float(h.iloc[back12]), 1), "pct_year": int(round(pctile[(g, a)].iloc[back12]))}
parts = [(g, a) for g in groups_e for a in bands if cells[g][a]]       # the groups that are measured (15 of 16)
(g1, a1), (g2, a2) = sorted(parts, key=lambda k: -cells[k[0]][k[1]]["pct"])[:2]
second = f"with {edu_short[g2]}" if a2 == a1 else f"{age_labels[a2]} year olds with {edu_short[g2]}"
whole = cells["all"]["15_64"]
reading_now = (f"All ages and education levels: {whole['rate']:.1f}% unemployed, the {ordinal(whole['pct'])} percentile of its "
               f"history since 2000. Furthest above normal: {age_labels[a1]} year olds with {edu_short[g1]} "
               f"({ordinal(cells[g1][a1]['pct'])}) and {second} ({ordinal(cells[g2][a2]['pct'])}).")
charts.append({
    "id": "cohorts_now", "title": "Which groups are worse off than usual?", "link": "cohorts", "wide": True,
    "reading": reading_now, "kind": "cohort_grids", "cells": cells,
    "edus": [{"key": g, "label": edu_labels[g], "short": edu_tab[g]} for g in edu_groups],
    # (all ages first, like the "All" column)
    "ages": [{"key": "15_64", "label": "All ages", "sub": age_labels["15_64"]}] + [{"key": a, "label": age_labels[a]} for a in bands],
    "month": month_name(lfs_month, short=True), "small": "too few to measure",
    "note": f"Unemployment rate by age and education, 12-month average to {month_name(lfs_month)}, not seasonally adjusted; "
            "percentile against each group's own history since January 2000.",
    **source("14-10-0019-01"),
})

# --- over time: every group's percentile, month by month since 2000 (the page's time range trims it) ---
months = [d.strftime("%Y-%m") for d in history[("all", "15_24")].index]
hist = {g: {a: None if history[(g, a)] is None else {"pct": [int(round(v)) for v in pctile[(g, a)]],
                                                     "rate": [round(float(v), 1) for v in history[(g, a)]]}
            for a in bands} for g in groups_e}
high = sum((pctile[k] >= 75).astype(int) for k in parts)            # groups at the 75th percentile or above, each month
peak, n_now = high.idxmax(), int(high.iloc[-1])
how_many = f"all {len(parts)}" if high[peak] == len(parts) else f"{high[peak]} of {len(parts)}"
reading_time = (f"In {month_name(peak)}, {how_many} groups were at the 75th percentile or above, the most since 2000; "
                f"in {month_name(lfs_month)}, {n_now} {'is' if n_now == 1 else 'are'}.")
charts.append({
    "id": "cohorts_time", "title": "How unusual has each group's unemployment been?", "link": "cohorts", "wide": True,
    "reading": reading_time, "kind": "cohort_heatmap", "months": months, "history": hist,
    "edus": [{"key": g, "label": edu_labels[g], "short": edu_tab[g]} for g in groups_e],
    "ages": [{"key": a, "label": age_labels[a]} for a in bands],
    "small": "too few to measure",
    "note": "Each group's 12-month average unemployment rate as a percentile of its own history since January 2000; "
            "not seasonally adjusted.",
    **source("14-10-0019-01"),
})

sections.append({"title": "Who is unemployed?", "charts": charts})
print(sum(len(s["charts"]) for s in sections), "charts in", len(sections), "sections")
for s in sections:
    for c in s["charts"]:
        print(f"[{s['title']}] {c['title']}\n    {c['reading']}")

## 8. Checks before writing

If any of these fail, the notebook stops and the old `labour.json` stays in place, so the page never shows broken data.
- the LFS is recent (published within the last 100 days of the month it covers),
- the headline numbers are in plausible ranges,
- the three parts of the unemployment breakdown add up to its change,
- every chart has data.

In [ ]:
today = pd.Timestamp.today()
assert (today - lfs_month).days < 100, "The latest LFS month is too old: the download may have failed."
assert 2 < now["unemp_rate"] < 20, "Unemployment rate out of range"
assert 70 < now["emp_rate_25_54"] < 95, "Prime-age employment rate out of range"
assert abs(jobs_change) < 500_000, "Monthly job change out of range"
for name in breakdown:
    b = breakdown[name].loc["1977":"2019"]
    assert (b[["jobs", "population", "participation"]].sum(axis=1) - b["total"]).abs().max() < 0.05, f"{name} breakdown does not add up"
assert sum(p is not None for p in pctile.values()) >= 18, "Too many age and education groups missing"
for section in sections:
    for chart in section["charts"]:
        has_data = chart.get("series") or chart.get("rows") or chart.get("cells") or chart.get("history")
        assert has_data, f"Chart {chart['id']} has no data"
print("All checks passed.")

## 9. Write the file

Everything goes into `../site/data/labour.json`.

The `headline` part also goes into a small file of its own, `../site/tile.json`: the labour tile on the Monitor
front page (panahovf.github.io/monitor/) reads it each time that page is opened, so the tile shows the latest
numbers without the front page being rebuilt. It has no build time in it, so it changes only when a number does.

In [ ]:
released = datetime.strptime(release_time["unemp_rate"], "%Y-%m-%dT%H:%M")

page = {
    "topic": "labour",
    "built_at": datetime.utcnow().strftime("%Y-%m-%dT%H:%MZ"),
    "updated": {"survey": "Labour Force Survey", "month": month_name(lfs_month),
                "released": f"{released.day} {released.strftime('%B %Y')}"},
    "headline": {"question": "How healthy is Canada's labour market?",
                 "value": f"{now['unemp_rate']:.1f}%", "label": "unemployment rate",
                 "change": f"{signed(ur_change)} pts in {lfs_month.strftime('%B')}",
                 "spark": spark(w["unemp_rate"])},
    "tiles": tiles,
    "changes": changes,
    "sections": sections,
}

with open("../site/data/labour.json", "w", encoding="utf-8") as f:
    json.dump(page, f, ensure_ascii=False, separators=(",", ":"))

print("Wrote ../site/data/labour.json")

# the labour tile on the Monitor front page: the headline, when the data were published, and where the page is
tile = dict(page["headline"], updated=page["updated"], url="https://panahovf.github.io/labour-monitor/")
with open("../site/tile.json", "w", encoding="utf-8") as f:
    json.dump(tile, f, ensure_ascii=False, separators=(",", ":"))

print("Wrote ../site/tile.json:", tile["value"], tile["label"], "·", tile["change"])